# 09 Audio + Engineered Lyrics Similarity

This notebook extends the audio-based similarity model by adding engineered lyrics features.

The previous notebook used only audio-related features as the baseline recommender.

In this notebook, lyrics-derived variables are added to test whether textual structure improves the quality of content-based recommendations.

## Notebook Goal

The goal of this notebook is to build and evaluate a second similarity-based recommendation model.

Model B combines:

- Spotify audio features
- Extracted low-level audio descriptors
- Engineered lyrics features

The results will later be compared against Model A, the audio-only baseline.

In [3]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / "data" / "processed"

tracks = pd.read_parquet(processed_dir / "tracks_clean.parquet")
artists = pd.read_parquet(processed_dir / "artists_clean.parquet")
albums = pd.read_parquet(processed_dir / "albums_clean.parquet")
audio_features = pd.read_parquet(processed_dir / "audio_features_clean.parquet")
lyrics_features = pd.read_parquet(processed_dir / "lyrics_features_valid_clean.parquet")

print(f"Tracks: {len(tracks):,}")
print(f"Artists: {len(artists):,}")
print(f"Albums: {len(albums):,}")
print(f"Audio Features: {len(audio_features):,}")
print(f"Lyrics Features: {len(lyrics_features):,}")

Tracks: 95,977
Artists: 56,129
Albums: 75,511
Audio Features: 95,948
Lyrics Features: 74,128


In [4]:
model_a_features = pd.read_csv(
    processed_dir / "model_a_audio_features.csv"
)["feature"].tolist()

print(f"Model A Audio Features: {len(model_a_features):,}")

Model A Audio Features: 206


# 1. Model B Feature Space

Model B builds on the audio-based baseline from the previous notebook.

The audio feature list is reused to ensure that the comparison between Model A and Model B is consistent.

The only difference is that Model B additionally includes engineered lyrics features.

In [5]:
lyrics_features.head()

,mean_syllables_word,mean_words_sentence,n_sentences,n_words,sentence_similarity,track_id,vocabulary_wealth
0,1.10,5.65,31,326,0.043011,13keyz9ikBe6ZpRasw7l4X,0.45
1,1.37,4.77,74,532,0.050352,1WugzepXsLjnsM0K4UaWYc,0.59
2,1.95,3.38,72,430,0.028560,2MO6oEAlMKcsfI8xP3yoy8,0.49
3,1.16,2.99,68,368,0.047849,1i4St7fmSUE9nB3R9n8fol,0.47
4,1.32,4.21,39,256,0.040486,3UyfvY3Gs6d4wvq8O4ANqQ,0.60


In [6]:
lyrics_feature_columns = [
    col
    for col in lyrics_features.columns
    if col != "track_id"
]

lyrics_feature_columns

['mean_syllables_word',
 'mean_words_sentence',
 'n_sentences',
 'n_words',
 'sentence_similarity',
 'vocabulary_wealth']

In [7]:
model_b_feature_summary = pd.DataFrame({
    "Feature Group": [
        "Audio Features",
        "Engineered Lyrics Features",
        "Total"
    ],
    "Count": [
        len(model_a_features),
        len(lyrics_feature_columns),
        len(model_a_features) + len(lyrics_feature_columns)
    ]
})

model_b_feature_summary

,Feature Group,Count
0,Audio Features,206
1,Engineered Lyrics Features,6
2,Total,212


# 2. Building the Model B Dataset

Model B requires tracks that have both audio features and engineered lyrics features.

This reduces the number of usable tracks compared to Model A, but adds information about lyrical structure and textual complexity.

In [8]:
model_b_master = (
    tracks
    .merge(
        audio_features,
        left_on="id",
        right_on="track_id",
        how="inner"
    )
    .merge(
        lyrics_features,
        left_on="id",
        right_on="track_id",
        how="inner",
        suffixes=("", "_lyrics")
    )
)

model_b_master.shape

(74103, 245)

In [9]:
print(f"Tracks available for Model B: {len(model_b_master):,}")
print(f"Feature count for Model B: {len(model_a_features) + len(lyrics_feature_columns)}")

Tracks available for Model B: 74,103
Feature count for Model B: 212


In [10]:
model_b_master[
    [
        "name",
        "popularity",
        "danceability",
        "energy",
        "valence",
        "tempo"
    ] + lyrics_feature_columns
].head()

,name,popularity,danceability,energy,valence,tempo,mean_syllables_word,mean_words_sentence,n_sentences,n_words,sentence_similarity,vocabulary_wealth
0,Blood,28.0,0.698,0.606,0.6220,115.018,1.39,3.13,39,208,0.028340,0.64
1,Already Gone,45.0,0.367,0.349,0.1920,81.850,1.49,3.17,24,133,0.021739,0.70
2,Creature Kind,47.0,0.748,0.666,0.3590,114.982,1.18,4.30,46,331,0.068599,0.53
3,Greatest Comedian,36.0,0.801,0.610,0.9420,104.199,1.26,3.38,40,220,0.055128,0.55
4,I Wish I Was A Shark,32.0,0.515,0.351,0.0383,139.926,1.42,2.89,38,239,0.035562,0.63


In [11]:
model_b_features = model_a_features + lyrics_feature_columns

X_model_b = model_b_master[model_b_features].copy()

print(f"Model B feature matrix shape: {X_model_b.shape}")
print(f"Missing values: {X_model_b.isna().sum().sum()}")

Model B feature matrix shape: (74103, 212)
Missing values: 0


## Model B Coverage Trade-Off

Compared to the audio-only baseline, Model B includes six engineered lyrics features describing textual structure and vocabulary usage.

This reduces the available track count from 95,948 to 74,103 tracks because lyrics-derived features are not available for every song.

However, the remaining dataset still provides substantial coverage while incorporating additional information about lyrical characteristics.

# 3. Feature Scaling

The audio and lyrics features operate on different numerical scales.

For example:

- Vocabulary wealth ranges between 0 and 1
- Number of words can exceed several hundred
- Audio descriptors use entirely different ranges

Therefore, all variables are standardized before similarity calculations.

In [12]:
from sklearn.preprocessing import StandardScaler

scaler_b = StandardScaler()

X_model_b_scaled = scaler_b.fit_transform(
    X_model_b
)

print(X_model_b_scaled.shape)

(74103, 212)


In [13]:
scaled_summary_b = pd.DataFrame({
    "Mean": X_model_b_scaled.mean(axis=0),
    "Std": X_model_b_scaled.std(axis=0)
})

scaled_summary_b.describe().round(3)

,Mean,Std
count,212.0,212.0
mean,0.0,1.0
std,0.0,0.0
min,-0.0,1.0
25%,-0.0,1.0
50%,0.0,1.0
75%,0.0,1.0
max,0.0,1.0


# 4. Similarity Computation

The same nearest-neighbor approach used in Model A is applied to the combined audio and lyrics feature space.

This allows direct comparison between both recommendation models.

In [14]:
from sklearn.preprocessing import StandardScaler

# Reset index for stable row lookups
model_b_master = model_b_master.reset_index(drop=True)

# Use ALL Model B features
X_model_b = model_b_master[
    model_b_features
].copy()

# Scale all features
scaler_b = StandardScaler()

X_model_b_scaled = scaler_b.fit_transform(
    X_model_b
)

print("Feature Matrix Shape:", X_model_b.shape)
print("Scaled Matrix Shape:", X_model_b_scaled.shape)

Feature Matrix Shape: (74103, 212)
Scaled Matrix Shape: (74103, 212)


In [15]:
from sklearn.neighbors import NearestNeighbors

nn_model_b = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11
)

nn_model_b.fit(X_model_b_scaled)

print("Model B fitted successfully.")

Model B fitted successfully.


In [ ]:
# save model and scaler for evaluation and reproducibility

import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(nn_model_b, "models/model_b.joblib")
joblib.dump(scaler_b, "models/scaler_b.joblib")

['models/scaler_b.joblib']

: 

In [14]:
print("Audio Features:", len(model_a_features))
print("Lyrics Features:", len(lyrics_feature_columns))
print("Model B Features:", len(model_b_features))

Audio Features: 206
Lyrics Features: 6
Model B Features: 212


In [15]:
def recommend_audio_tracks_model_b(song_name, n_recommendations=10, remove_same_title=True):
    matches = model_b_master[
        model_b_master["name"].str.contains(song_name, case=False, na=False)
    ]

    if matches.empty:
        print(f"No song found for: {song_name}")
        return None

    # Use most popular matching version as query song
    query_idx = matches["popularity"].idxmax()
    query_title = model_b_master.loc[query_idx, "name"]

    distances, indices = nn_model_b.kneighbors(
        X_model_b_scaled[query_idx].reshape(1, -1),
        n_neighbors=n_recommendations + 20
    )

    recommendations = model_b_master.iloc[indices[0]].copy()
    recommendations["cosine_distance"] = distances[0]

    # Remove query song itself
    recommendations = recommendations[
        recommendations.index != query_idx
    ]

    # Optional: remove duplicate/same-title recommendations
    if remove_same_title:
        recommendations = recommendations[
            recommendations["name"].str.lower() != query_title.lower()
        ]

    result = recommendations[
        [
            "name",
            "popularity",
            "danceability",
            "energy",
            "valence",
            "tempo",
            "cosine_distance"
        ]
    ].head(n_recommendations)

    print(f"Recommendations for: {query_title}")

    return result

In [16]:
model_b_master["name"].sample(20, random_state=42).tolist()

['O Amanha',
 'Such a Remarkable Day',
 '40th Street Black / We Will Fight',
 'Off Road',
 'Honey',
 'Fresh Out',
 'Sold Dreams (Interlude) [feat. Hood Funny Ayo]',
 "It's Not the End",
 'Every Kind Of Way',
 'Rain Is Gone',
 'Peace - A Mushroom Cloud',
 'Break Loose (feat. NEIMY)',
 "Llamé Pa' Verte",
 'Dan Volg Je Haar Benen',
 'Trátame Suavemente - Remasterizado 2007',
 'Yasis',
 'María del Carmen',
 'Something Special - Original Mix',
 'Call My Name',
 'Turn']

In [17]:
recommend_audio_tracks_model_b("Billie Jean", n_recommendations=10)

Recommendations for: Billie Jean


,name,popularity,danceability,energy,valence,tempo,cosine_distance
50066,Michael Jackson x Mark Ronson: Diamonds are In...,59.0,0.672,0.832,0.549,117.298,0.251270
42382,Hooligan,36.0,0.612,0.898,0.782,150.030,0.260754
12201,Never Been In Love (feat. Icona Pop),47.0,0.631,0.929,0.695,121.080,0.265919
74063,Hate Being Alone,55.0,0.501,0.969,0.170,74.767,0.267909
74059,Pressure - Blanke Remix,44.0,0.512,0.944,0.280,187.916,0.271784
49104,Camo Diamond Rollie,52.0,0.604,0.984,0.139,145.095,0.274182
32433,Lightspeed,45.0,0.670,0.977,0.543,159.944,0.274303
59135,Get Low,48.0,0.844,0.752,0.580,97.005,0.275268
62314,Wild Thoughts - Dave Audé Dance Remix,53.0,0.698,0.924,0.641,116.040,0.275582
535,Mission,40.0,0.817,0.772,0.373,87.507,0.276252


In [18]:
recommend_audio_tracks_model_b("Starboy", n_recommendations=10)

Recommendations for: Starboy - Acoustic


,name,popularity,danceability,energy,valence,tempo,cosine_distance
29278,With U 2,35.0,0.680,0.2890,0.6760,78.528,0.265189
30967,You Are Not Alone,43.0,0.753,0.2080,0.2810,115.009,0.284284
73645,Here Today,34.0,0.533,0.1160,0.0391,75.041,0.310030
18082,Touch Me,40.0,0.944,0.3820,0.5020,105.043,0.312762
9530,Next to You,50.0,0.443,0.3320,0.0689,87.839,0.314751
20520,Génie,53.0,0.548,0.4330,0.1530,75.485,0.316959
5622,leave me behind,32.0,0.515,0.0402,0.0830,90.779,0.319133
13577,Let Me Go,46.0,0.563,0.4230,0.0839,92.882,0.320472
43990,Loco,60.0,0.746,0.3810,0.2470,100.049,0.323651
50994,En attendant demain - Single,24.0,0.602,0.3340,0.2330,81.809,0.326702


In [19]:
recommend_audio_tracks_model_b("Time", n_recommendations=10)

Recommendations for: Sign of the Times


,name,popularity,danceability,energy,valence,tempo,cosine_distance
64103,It Takes A Fool To Remain Sane,56.0,0.692,0.585,0.3860,122.955,0.384683
24850,Better In The Dark,53.0,0.670,0.758,0.4920,110.024,0.460750
46111,Honest,44.0,0.419,0.541,0.1730,147.416,0.468230
47224,Un Amor Como El Nuestro,37.0,0.616,0.274,0.6460,175.954,0.476028
51662,Not Too Late,55.0,0.657,0.693,0.2710,120.041,0.487496
39820,Not Too Late,49.0,0.657,0.693,0.2710,120.041,0.493934
68056,Just Goes to Show,34.0,0.493,0.807,0.4080,100.246,0.495108
45142,Princess of China - Radio Edit,61.0,0.402,0.700,0.3140,85.025,0.497676
11445,Always Have - Live,37.0,0.266,0.366,0.0668,93.336,0.498692
65254,Say Something,78.0,0.707,0.632,0.3720,97.040,0.501170


In [20]:
recommend_audio_tracks_model_b("Like A Prayer", n_recommendations=10)

Recommendations for: Like a Prayer


,name,popularity,danceability,energy,valence,tempo,cosine_distance
42654,Sk8er Boi,76.0,0.487,0.900,0.4840,149.937,0.291280
29816,The Resistance,69.0,0.483,0.941,0.5030,156.033,0.300450
11802,Still Into You,73.0,0.602,0.923,0.7650,136.010,0.301174
71391,All Out Life,74.0,0.499,0.970,0.0506,106.558,0.301860
6286,SING,61.0,0.606,0.942,0.3730,110.980,0.310342
56219,Be Alright,21.0,0.451,0.796,0.4920,94.986,0.333420
21705,Ready to Go,51.0,0.553,0.954,0.2350,130.046,0.334486
41652,In Between,64.0,0.462,0.972,0.4490,128.092,0.334810
23899,Die MF Die,69.0,0.657,0.960,0.5670,126.020,0.334965
60232,Ready to Go,51.0,0.559,0.952,0.2430,130.047,0.337719


In [21]:
print(len(model_a_features))
print(len(model_b_features))

206
212


### Model B First Result Interpretation

After adding engineered lyrics features, the recommendation results remain broadly similar to the audio-only baseline.

This suggests that the six lyrics-derived variables influence the similarity calculation only moderately compared to the much larger audio feature space.

The added lyrics features describe structural properties of lyrics, such as word count, sentence count, vocabulary wealth, and sentence similarity. They do not capture semantic meaning, themes, emotion, or topic.

Therefore, Model B is useful as an intermediate experiment, but it is not expected to strongly change the recommendation logic. A stronger lyrics-based model would require using the full lyrics text, for example through TF-IDF vectorization.

In [22]:
def recommendation_distance_summary_b(query_names, n_recommendations=10):
    summaries = []

    for song_name in query_names:
        recs = recommend_audio_tracks_model_b(
            song_name,
            n_recommendations=n_recommendations
        )

        if recs is not None:
            summaries.append({
                "Query Track": song_name,
                "Mean Distance": recs["cosine_distance"].mean(),
                "Min Distance": recs["cosine_distance"].min(),
                "Max Distance": recs["cosine_distance"].max()
            })

    return pd.DataFrame(summaries)

In [23]:
query_names = [
    "Billie Jean",
    "Starboy",
    "Sign of the Times",
    "Like a Prayer"
]

model_b_distance_summary = recommendation_distance_summary_b(
    query_names
)

model_b_distance_summary

Recommendations for: Billie Jean
Recommendations for: Starboy - Acoustic


Recommendations for: Sign of the Times
Recommendations for: Like a Prayer


,Query Track,Mean Distance,Min Distance,Max Distance
0,Billie Jean,0.269322,0.251270,0.276252
1,Starboy,0.309393,0.265189,0.326702
2,Sign of the Times,0.476377,0.384683,0.501170
3,Like a Prayer,0.318051,0.291280,0.337719


# 5. Model B Evaluation

To assess the impact of the engineered lyrics features, recommendation distances were summarized across multiple example tracks.

The results indicate that recommendations remain broadly similar to those produced by the audio-only baseline.

This outcome is expected because the additional lyrics features describe structural characteristics of the lyrics rather than semantic meaning. While they contribute information about vocabulary usage, sentence structure, and lyrical complexity, they do not capture themes, topics, sentiment, or contextual meaning.

Consequently, Model B provides only a modest extension of the audio-based similarity model.

### Model B Conclusion

Model B successfully integrates engineered lyrics features into the recommendation pipeline while maintaining a reasonably large dataset coverage of approximately 77%.

However, the recommendation results remain largely driven by the audio feature space, suggesting that the engineered lyrics variables contribute only limited additional information.

The next step is therefore to incorporate the full lyrics text and investigate whether semantic information extracted directly from song lyrics can improve recommendation quality.

In [24]:
model_b_feature_path = (
    processed_dir /
    "model_b_audio_lyrics_features.csv"
)

pd.Series(
    model_b_features,
    name="feature"
).to_csv(
    model_b_feature_path,
    index=False
)

print(
    f"Saved Model B feature list to: "
    f"{model_b_feature_path}"
)

Saved Model B feature list to: /workspaces/spotify_project/data/processed/model_b_audio_lyrics_features.csv


In [ ]:
model_b_summary = pd.DataFrame({
    "Metric": [
        "Tracks",
        "Features",
        "Audio Features",
        "Lyrics Features"
    ],
    "Value": [
        len(model_b_master),
        len(model_b_features),
        len(model_a_features),
        len(lyrics_feature_columns)
    ]
})

model_b_summary.to_csv(
    processed_dir /
    "model_b_summary.csv",
    index=False
)

model_b_summary

,Metric,Value
0,Tracks,74103
1,Features,212
2,Audio Features,206
3,Lyrics Features,6


: 